In [1]:
pip install mne

^C
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement mne (from versions: none)
ERROR: No matching distribution found for mne


in this part I imported the data, defined pubishment and achivment trials ,selected only Cz,Pz and Fz channels and calculated Z-score and SE.

In [ ]:
import mne
import numpy as np
import matplotlib.pyplot as plt

def zscore(x):
    return (x - np.mean(x)) / np.std(x)
def SE(x):
    return (np.std(x)) / np.sqrt(x)

epochs = mne.io.read_epochs_eeglab('594 cleaned.set')

selected_channels = ['FZ', 'CZ', 'PZ']
# Note: The pick_channels() function is legacy. New code should use epochs.pick(...)
epochs_selected = epochs.copy().pick_channels(selected_channels)

epochs_zscored = epochs_selected.copy().apply_function(zscore)

evoked = epochs_zscored.average()
evoked.plot()

epochs=epochs_zscored

punishment_keys = [k for k in epochs.event_id if '/94' in k]
achievement_keys = [k for k in epochs.event_id if '/104' in k]

# Select channels of interest: FZ and CZ
epochs.pick(['FZ', 'CZ'])

# Extract epochs for each event type using the event labels
punishment_epochs = epochs[punishment_keys]
achievement_epochs = epochs[achievement_keys]

# Select the first five epochs (trials) for each event type
punishment_epochs_5 = punishment_epochs[:10]
achievement_epochs_5 = achievement_epochs[:10]

# For each event type, pick the channel FZ
punishment_FZ = punishment_epochs_5.copy().pick_channels(['FZ'])
achievement_FZ = achievement_epochs_5.copy().pick_channels(['FZ'])

punishment_CZ = punishment_epochs_5.copy().pick_channels(['CZ'])
achievement_CZ = achievement_epochs_5.copy().pick_channels(['CZ'])

# Compute the averaged evoked responses
evoked_punishment_FZ = punishment_FZ.average()
evoked_achievement_FZ = achievement_FZ.average()

evoked_punishment_CZ = punishment_CZ.average()
evoked_achievement_CZ = achievement_CZ.average()

# Create a dictionary to compare evoked responses
compare_dict = {
    'Punishment': evoked_punishment_FZ,
    'Achievement': evoked_achievement_FZ
}

# Plot the comparison for channel FZ
mne.viz.plot_compare_evokeds(compare_dict, picks=['FZ'], title='FZ: Punishment vs. Achievement DEPRESSED SUBJECT', show=True)

compare_dict = {
    'Punishment': evoked_punishment_CZ,
    'Achievement': evoked_achievement_CZ
}

mne.viz.plot_compare_evokeds(compare_dict, picks=['CZ'], title='CZ: Punishment vs. Achievement DEPRESSED SUBJECT', show=True)



code for ERP calculation part:


In [ ]:
import mne
import numpy as np
import matplotlib.pyplot as plt

# === Define participant groups ===
cont_files = ['513_cleaned.set', '568 cleaned.set', '532 cleaned.set', '529 cleaned.set', '534 cleaned.set'] 

# === define function for zscore and se === #
def zscore(x):
    return (x - np.mean(x)) / np.std(x)
def SE(x,n):
    return (np.std(x)) / np.sqrt(n)

# === define function for recognition of reward === #
def load_a(files, label):
    evokeds = []
    min_len = None
    counts=[]
    for file in files:
        print(f'Loading {file}...')

        try:
            epochs = mne.read_epochs_eeglab(file)
            epochs.pick_channels(['FZ', 'CZ', 'PZ'])
            epochs = epochs.copy().apply_function(zscore)
            selected = epochs[['keypad2/104', 'keypad1/104', 'keypad1/keypad1/104', 'keypad2/keypad2/104']]
        except Exception as e:
            print(f"⚠️ Skipping {file}: {e}")
            continue

        if len(selected) == 0:
            print(f"⚠️ No usable epochs in {file}")
            continue

        data = selected.get_data().mean(axis=0)
        counts.append(len(selected))

        if min_len is None or data.shape[1] < min_len:
            min_len = data.shape[1]

        evokeds.append(data)

    if len(evokeds) == 0:
        raise ValueError(f"❌ No usable data in {label}")

    # Trim all to shortest length
    evokeds = [e[:, :min_len] for e in evokeds]
    return np.stack(evokeds), selected.times[:min_len] , counts # return data and trimmed time vector


def load_p(files, label):
    evokeds = []
    min_len = None
    counts=[]
    for file in files:
        print(f'Loading {file}...')

        try:
            epochs = mne.read_epochs_eeglab(file)
            epochs.pick_channels(['FZ', 'CZ', 'PZ'])
            epochs = epochs.copy().apply_function(zscore)
            selected = epochs[['keypad2/94', 'keypad1/94', 'keypad1/keypad1/94', 'keypad2/keypad2/94']]
        except Exception as e:
            print(f"⚠️ Skipping {file}: {e}")
            continue

        if len(selected) == 0:
            print(f"⚠️ No usable epochs in {file}")
            continue

        data = selected.get_data().mean(axis=0)
        counts.append(len(selected))
        if min_len is None or data.shape[1] < min_len:
            min_len = data.shape[1]

        evokeds.append(data)

    if len(evokeds) == 0:
        raise ValueError(f"❌ No usable data in {label}")

    # Trim all to shortest length
    evokeds = [e[:, :min_len] for e in evokeds]
    return np.stack(evokeds), selected.times[:min_len] , counts # return data and trimmed time vector


# === Load both groups ===

data2, times2, counts2 = load_a(cont_files, "control")
data4, times4, counts4 = load_p(cont_files, "control")



# === Compute grand averages ===
grand_avg2 = data2.mean(axis=0) # shape: (n_channels, n_times)
grand_avg4 = data4.mean(axis=0)

# standard error across subjects (axis=0)
se2 = data2.std(axis=0) / np.sqrt(len(data2))
se4 = data4.std(axis=0) / np.sqrt(len(data4))


# === Plot CZ channel from both groups ===
ch_names = ['FZ', 'CZ', 'PZ']
ch = ch_names.index("CZ")

plt.plot(times2, grand_avg2[ch, :], label='Reward', color='blue', linewidth=2)
plt.fill_between(times2,
                 grand_avg2[ch, :] - se2[ch, :],
                 grand_avg2[ch, :] + se2[ch, :],
                 color='blue', alpha=0.3)

plt.plot(times4, grand_avg4[ch, :], label='Punishment', color='red', linewidth=2)
plt.fill_between(times4,
                 grand_avg4[ch, :] - se4[ch, :],
                 grand_avg4[ch, :] + se4[ch, :],
                 color='red', alpha=0.3)

plt.xlabel('Time (ms)')
plt.ylabel('Amplitude (µV)')
plt.title('ERP Comparison at CZ Control subjects')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


This code creates a 4D tensor

Subject×Trial×Frequency×Time

by applying STFT to the FZ channel of each EEG trial, computes spectral power ∣STFT∣2, averages across subjects and trials, and then compares the average spectrograms of healthy and depressed groups. no baseline normalization (ERSP) is performed.

In [ ]:

import mne
import numpy as np
import os
from scipy.signal import stft
import warnings
import matplotlib.pyplot as plt

# غیرفعال کردن هشدارها
warnings.filterwarnings('ignore')

# تنظیمات اصلی
data_dir = r"C:\Users\Sons"
output_dir = r'C:\Users\Sons'
os.makedirs(output_dir, exist_ok=True)

# پارامترهای پردازش
TARGET_CHANNEL = 'FZ'  # کانال هدف
FS = 250               # فرکانس نمونه‌برداری (بررسی شود)
NPERSEG = 128          # طول هر قطعه برای STFT (کاهش یافته)
NOVERLAP = 64          # همپوشانی برای STFT
NTRIALS = 15           # تعداد آزمون‌ها از هر سوژه

# لیست فایل‌ها
set_files = [f for f in os.listdir(data_dir) if f.endswith('.set')]

# دسته‌بندی سوژه‌ها
healthy_ids = ['513', '568', '532' ,'529', '534']                                # سوژه‌های سالم
depressed_ids = ['598', '558', '587', '594', '597']  # سوژه‌های افسرده

def extract_subject_id(filename):
    """استخراج شناسه سوژه از نام فایل"""
    return filename.split()[0]

def load_and_process(file_path):
    """بارگذاری و پردازش هر فایل EEG"""
    try:
      
        try:
            data = mne.io.read_epochs_eeglab(file_path)
            data_type = 'epochs'
        except:
            data = mne.io.read_raw_eeglab(file_path, preload=True)
            data_type = 'raw'
        
        # انتخاب کانال
        selected_channel = TARGET_CHANNEL if TARGET_CHANNEL in data.ch_names else data.ch_names[0]
        ch_idx = data.ch_names.index(selected_channel)
        
        # استخراج داده‌ها
        if data_type == 'epochs':
            ch_data = data.get_data()[:, ch_idx, :]
        else:
            ch_data = data.get_data()[ch_idx, :]
        
        return {
            'sfreq': data.info['sfreq'],
            'data': ch_data,
            'type': data_type,
            'channel': selected_channel
        }
    except Exception as e:
        print(f"خطا در پردازش {file_path}: {str(e)}")
        return None

# پردازش تمام فایل‌ها
all_data = {}
for file in set_files:
    subject_id = extract_subject_id(file)
    file_path = os.path.join(data_dir, file)
    processed = load_and_process(file_path)
    if processed is not None:
        all_data[subject_id] = processed
        print(f"پردازش {file} با موفقیت انجام شد. کانال: {processed['channel']}")

def create_power_tensor(subjects_list, data_dict):
    """ساخت تانسور توان طیفی"""
    if not subjects_list:
        return None
    
    # تعیین ابعاد خروجی STFT
    _, _, Zxx_sample = stft(np.zeros(NPERSEG), fs=FS, nperseg=NPERSEG, noverlap=NOVERLAP)
    n_freq, n_time = Zxx_sample.shape
    
    # ساخت تانسور خالی
    tensor = np.zeros((len(subjects_list), NTRIALS, n_freq, n_time))
    
    for i, subj in enumerate(subjects_list):
        subject_data = data_dict[subj]
        data = subject_data['data']
        sfreq = subject_data['sfreq']
        
        if subject_data['type'] == 'epochs':
            # انتخاب تصادفی trials
            n_available = data.shape[0]
            trials = np.random.choice(n_available, size=min(NTRIALS, n_available), replace=False)
            
            for j, trial_idx in enumerate(trials):
                segment = data[trial_idx]
                # تطبیق طول داده
                if len(segment) < NPERSEG:
                    segment = np.pad(segment, (0, NPERSEG - len(segment)))
                else:
                    segment = segment[:NPERSEG]
                
                _, _, Zxx = stft(segment, fs=sfreq, nperseg=NPERSEG, noverlap=NOVERLAP)
                tensor[i,j] = np.abs(Zxx)**2
        else:
            # تقسیم داده raw به بخش‌های مساوی
            segment_length = NPERSEG
            n_segments = len(data) // segment_length
            n_segments = min(n_segments, NTRIALS)
            
            for j in range(n_segments):
                start = j * segment_length
                segment = data[start: start + segment_length]
                _, _, Zxx = stft(segment, fs=sfreq, nperseg=NPERSEG, noverlap=NOVERLAP)
                tensor[i,j] = np.abs(Zxx)**2
                
    return tensor


# ساخت تانسورها
healthy_subjects = [subj for subj in healthy_ids if subj in all_data]
depressed_subjects = [subj for subj in depressed_ids if subj in all_data]

healthy_tensor = create_power_tensor(healthy_subjects, all_data)
depressed_tensor = create_power_tensor(depressed_subjects, all_data)

# ذخیره نتایج
if healthy_tensor is not None:
    np.save(os.path.join(output_dir, 'healthy_power.npy'), healthy_tensor)
    print(f"\nتانسور سالم ذخیره شد. ابعاد: {healthy_tensor.shape}")

if depressed_tensor is not None:
    np.save(os.path.join(output_dir, 'depressed_power.npy'), depressed_tensor)
    print(f"تانسور افسرده ذخیره شد. ابعاد: {depressed_tensor.shape}")

# نمایش نتایج
def plot_power_comparison():
    """نمایش گرافیکی مقایسه توان طیفی"""
    if healthy_tensor is not None and depressed_tensor is not None:
        plt.figure(figsize=(15,6))
        
        plt.subplot(131)
        plt.imshow(healthy_tensor.mean(axis=(0,1)), aspect='auto', cmap='jet')
        plt.title('control subject ')
        plt.xlabel('time')
        plt.ylabel('frequency')
        plt.colorbar()
        
        plt.subplot(132)
        plt.imshow(depressed_tensor.mean(axis=(0,1)), aspect='auto', cmap='jet')
        plt.title('depressed subject')
        plt.xlabel('time')
        plt.colorbar()
        
        plt.subplot(133)
        diff = depressed_tensor.mean(axis=(0,1)) - healthy_tensor.mean(axis=(0,1))
        plt.imshow(diff, aspect='auto', cmap='coolwarm')
        plt.title('depressed mines control')
        plt.xlabel('time')
        plt.colorbar()
        
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'power_comparison.png'))
        plt.show()

plot_power_comparison()
print("\nپردازش کامل شد!")




in the following code I:
load EEGLAB epochs
extract FZ channel
compute STFT power
build subject × trial × frequency × time tensors
compare healthy vs depressed groups
visualize group differences

In [ ]:
import mne
import numpy as np
import os
from scipy.signal import stft
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

# مسیرها
data_dir = r"C:\Users\Sons"
output_dir = r'C:\Users\Sons'
os.makedirs(output_dir, exist_ok=True)

# تنظیمات عمومی
TARGET_CHANNEL = 'FZ'
FS = 250
NPERSEG = 128
NOVERLAP = 64
NTRIALS = 15

# شناسه سوژه‌ها
healthy_ids = ['513', '568', '532', '529', '534']
depressed_ids = ['598', '558', '587', '594', '597']

def extract_subject_id(filename):
    return filename.split()[0]

# --- توابع بارگذاری اپوک‌ها ---

def load_reward_epochs(file_path):
    """دریافت اپوک‌های مربوط به پاداش"""
    epochs = mne.read_epochs_eeglab(file_path)
    selected = epochs[['keypad2/104', 'keypad1/104', 'keypad1/keypad1/104', 'keypad2/keypad2/104']]
    
    if TARGET_CHANNEL not in selected.ch_names:
        raise ValueError(f"{TARGET_CHANNEL} not in {file_path}")
    
    ch_idx = selected.ch_names.index(TARGET_CHANNEL)
    data = selected.get_data()[:, ch_idx, :]
    return data, selected.info['sfreq']

def load_punishment_epochs(file_path):
    """دریافت اپوک‌های مربوط به مجازات"""
    epochs = mne.read_epochs_eeglab(file_path)
    selected = epochs[['keypad2/94', 'keypad1/94', 'keypad1/keypad1/94', 'keypad2/keypad2/94']]
    
    if TARGET_CHANNEL not in selected.ch_names:
        raise ValueError(f"{TARGET_CHANNEL} not in {file_path}")
    
    ch_idx = selected.ch_names.index(TARGET_CHANNEL)
    data = selected.get_data()[:, ch_idx, :]
    return data, selected.info['sfreq']

# --- پردازش STFT ---

def compute_stft_tensor(data, sfreq):
    _, _, Zxx_sample = stft(np.zeros(NPERSEG), fs=sfreq, nperseg=NPERSEG, noverlap=NOVERLAP)
    n_freq, n_time = Zxx_sample.shape
    n_epochs = min(NTRIALS, data.shape[0])
    
    tensor = np.zeros((n_epochs, n_freq, n_time))
    for i in range(n_epochs):
        segment = data[i]
        if len(segment) < NPERSEG:
            segment = np.pad(segment, (0, NPERSEG - len(segment)))
        else:
            segment = segment[:NPERSEG]
        _, _, Zxx = stft(segment, fs=sfreq, nperseg=NPERSEG, noverlap=NOVERLAP)
        tensor[i] = np.abs(Zxx)**2
    return tensor

# --- ساخت تانسورها برای گروه‌ها ---

def process_group(subject_ids, event_type):
    tensors = []
    for file in os.listdir(data_dir):
        if not file.endswith('.set'):
            continue
        subject_id = extract_subject_id(file)
        if subject_id not in subject_ids:
            continue
        try:
            file_path = os.path.join(data_dir, file)
            if event_type == 'reward':
                data, sfreq = load_reward_epochs(file_path)
            else:
                data, sfreq = load_punishment_epochs(file_path)
            tensor = compute_stft_tensor(data, sfreq)
            if tensor is not None:
                tensors.append(tensor)
                print(f"✓ {subject_id} ({event_type})")
        except Exception as e:
            print(f"✗ خطا در {subject_id} ({event_type}): {e}")
    try:
        result = np.stack(tensors)
        print(f"✅ تانسور {event_type} برای {len(tensors)} نفر ساخته شد. سایز: {result.shape}")
        return result
    except Exception as e:
        print(f"⛔ خطا در ساخت نهایی تانسور {event_type}: {e}")
        return None

# --- ساخت داده‌ها ---

reward_healthy = process_group(healthy_ids, 'reward')
reward_depressed = process_group(depressed_ids, 'reward')
punish_healthy = process_group(healthy_ids, 'punishment')
punish_depressed = process_group(depressed_ids, 'punishment')

# --- ذخیره داده‌ها ---

def save_tensor(tensor, name):
    if tensor is not None:
        path = os.path.join(output_dir, name + '.npy')
        np.save(path, tensor)
        print(f"💾 ذخیره شد: {path} — شکل: {tensor.shape}")

save_tensor(reward_healthy, 'reward_healthy')
save_tensor(reward_depressed, 'reward_depressed')
save_tensor(punish_healthy, 'punish_healthy')
save_tensor(punish_depressed, 'punish_depressed')

# --- رسم نمودارها ---

def plot_tensor_diff(tensor_1, tensor_2, title1, title2, suptitle, filename):
    if tensor_1 is None or tensor_2 is None:
        print(f"⛔ داده‌ای برای {suptitle} وجود ندارد.")
        return
    
    avg_1 = tensor_1.mean(axis=(0,1))
    avg_2 = tensor_2.mean(axis=(0,1))
    diff = avg_2 - avg_1

    plt.figure(figsize=(15, 5))

    plt.subplot(131)
    plt.imshow(avg_1, aspect='auto', cmap='jet')
    plt.title(title1)
    plt.xlabel('Time')
    plt.ylabel('Frequency')
    plt.colorbar()

    plt.subplot(132)
    plt.imshow(avg_2, aspect='auto', cmap='jet')
    plt.title(title2)
    plt.xlabel('Time')
    plt.colorbar()

    plt.subplot(133)
    plt.imshow(diff, aspect='auto', cmap='bwr')
    plt.title(f'Diff: {title2} - {title1}')
    plt.xlabel('Time')
    plt.colorbar()

    plt.suptitle(suptitle)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, filename))
    plt.show()

plot_tensor_diff(
    reward_healthy, reward_depressed,
    'Reward - Healthy', 'Reward - Depressed',
    'Power Changes - Reward',
    'reward_comparison.png'
)

plot_tensor_diff(
    punish_healthy, punish_depressed,
    'Punishment - Healthy', 'Punishment - Depressed',
    'Power Changes - Punishment',
    'punishment_comparison.png'
)

print("\n🎉 پردازش کامل شد.")


ERSP:

In [ ]:
import mne
import numpy as np
import matplotlib.pyplot as plt

# === فایل‌های EEG شرکت‌کننده‌ها ===
files = ['594 cleaned.set', '598 cleaned.set', '587 cleaned.set', '597 cleaned.set', '558 cleaned.set']

# === نوع رویدادها برای پاداش و مجازات ===
reward_ids = ['keypad2/104', 'keypad1/104', 'keypad1/keypad1/104', 'keypad2/keypad2/104']
punish_ids = ['keypad2/94', 'keypad1/94', 'keypad1/keypad1/94', 'keypad2/keypad2/94']

# === پارامترهای تبدیل زمان-فرکانس (TFR) ===
frequencies = np.linspace(4, 40, 30)
n_cycles = frequencies / 2.  # تعداد سیکل‌ها برای موجک مورله

# === لیست برای ذخیره قدرت ERSP ===
reward_powers = []
punish_powers = []

# === حلقه پردازش هر فایل ===
for file in files:
    print(f"Processing {file}")
    epochs = mne.read_epochs_eeglab(file, preload=True)

    # نرمال‌سازی z-score در زمان برای هر اپک
    data = epochs.get_data()
    zscored = (data - data.mean(axis=2, keepdims=True)) / data.std(axis=2, keepdims=True)
    epochs._data = zscored  # هشدار: تغییر مستقیم داده‌ها

    # استخراج اپک‌های مرتبط با پاداش و مجازات
    try:
        reward_epochs = epochs[reward_ids]
        punish_epochs = epochs[punish_ids]
    except Exception as e:
        print(f"⚠️ Skipping {file} due to missing events: {e}")
        continue

    # انتخاب ۱۵ اپک تصادفی (در صورت کافی بودن)
    rng = np.random.default_rng(seed=42)
    if len(reward_epochs) >= 15:
        reward_epochs = reward_epochs[rng.choice(len(reward_epochs), 15, replace=False)]
    if len(punish_epochs) >= 15:
        punish_epochs = punish_epochs[rng.choice(len(punish_epochs), 15, replace=False)]

    # محاسبه توان ERSP با موجک مورله بدون میانگین‌گیری
    reward_power = mne.time_frequency.tfr_morlet(
        reward_epochs, freqs=frequencies, n_cycles=n_cycles,
        use_fft=True, return_itc=False, average=False
    )
    punish_power = mne.time_frequency.tfr_morlet(
        punish_epochs, freqs=frequencies, n_cycles=n_cycles,
        use_fft=True, return_itc=False, average=False
    )

    # میانگین‌گیری درون‌فردی برای تبدیل به AverageTFR
    reward_power_avg = reward_power.average()
    punish_power_avg = punish_power.average()

    # نرمال‌سازی با پایه‌ی قبل از رویداد (۵۰۰- میلی‌ثانیه تا ۰)
    reward_power_avg.apply_baseline(baseline=(-0.5, 0), mode='logratio')
    punish_power_avg.apply_baseline(baseline=(-0.5, 0), mode='logratio')

    reward_powers.append(reward_power_avg)
    punish_powers.append(punish_power_avg)

# === میانگین گروهی (grand average) ===
reward_avg = mne.grand_average(reward_powers)
punish_avg = mne.grand_average(punish_powers)

# === استخراج داده کانال CZ برای نمایش ===
ch_index = reward_avg.info['ch_names'].index('CZ')
reward_cz = reward_avg.data[ch_index]
punish_cz = punish_avg.data[ch_index]

# === رسم نمودار ERSP در کانال CZ ===
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(reward_cz, aspect='auto', origin='lower',
           extent=[reward_avg.times[0], reward_avg.times[-1], frequencies[0], frequencies[-1]])
plt.title('Reward ERSP at CZ')
plt.xlabel('Time (s)')
plt.ylabel('Frequency (Hz)')
plt.colorbar(label='Power Ratio')

plt.subplot(1, 2, 2)
plt.imshow(punish_cz, aspect='auto', origin='lower',
           extent=[punish_avg.times[0], punish_avg.times[-1], frequencies[0], frequencies[-1]])
plt.title('Punishment ERSP at CZ')
plt.xlabel('Time (s)')
plt.ylabel('Frequency (Hz)')
plt.colorbar(label='Power Ratio')

plt.tight_layout()
plt.show()



find full code explnation here:
https://chatgpt.com/share/6a2052a7-fb84-83eb-a1cc-5c33b3fea101
